In [13]:
import pandas as pd
import datetime
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# make network graph
import plotly.graph_objects as go
import networkx as nx
import matplotlib.pyplot as plt

In [22]:
IDS_df1 = pd.read_csv('DC4-data/IDS/IDS-0406.csv')
IDS_df2 = pd.read_csv('DC4-data/IDS/IDS-0407.csv')
IDS_df = pd.concat([IDS_df1, IDS_df2])
IDS_df['Date/time'] = pd.to_datetime(IDS_df['time'])
IDS_df.head()

,time,sourceIP,sourcePort,destIP,destPort,classification,priority,label,packet info,packet info cont'd,xref,label,packet info,packet info cont'd,Date/time
0,4/5/2012 17:55,172.23.1.101,1101,172.23.0.10,139,Generic Protocol Command Decode,3,[1:2100538:17] GPL NETBIOS SMB IPC$ unicode s...,TCP TTL:128 TOS:0x0 ID:1643 IpLen:20 DgmLen:12...,***AP*** Seq: 0xCEF93F32 Ack: 0xC40C0BB Win:...,NaN,NaN,NaN,NaN,2012-04-05 17:55:00
1,4/5/2012 17:55,172.23.1.101,1101,172.23.0.10,139,Generic Protocol Command Decode,3,[1:2100538:17] GPL NETBIOS SMB IPC$ unicode s...,TCP TTL:128 TOS:0x0 ID:1649 IpLen:20 DgmLen:12...,***AP*** Seq: 0xCEF942A6 Ack: 0xC40C427 Win:...,NaN,NaN,NaN,NaN,2012-04-05 17:55:00
2,4/5/2012 17:55,172.23.1.101,1104,172.23.0.10,139,Generic Protocol Command Decode,3,[1:2103000:7] GPL NETBIOS SMB Session Setup N...,TCP TTL:128 TOS:0x0 ID:1663 IpLen:20 DgmLen:15...,***A**** Seq: 0x54FEF4C7 Ack: 0x9BEB6342 Win...,[Xref => http://www.microsoft.com/technet/secu...,NaN,NaN,NaN,2012-04-05 17:55:00
3,4/5/2012 17:55,172.23.1.101,1104,172.23.0.10,139,Generic Protocol Command Decode,3,[1:2100538:17] GPL NETBIOS SMB IPC$ unicode s...,TCP TTL:128 TOS:0x0 ID:1665 IpLen:20 DgmLen:12...,***AP*** Seq: 0x54FEFF59 Ack: 0x9BEB64C7 Win...,NaN,NaN,NaN,NaN,2012-04-05 17:55:00
4,4/5/2012 17:56,172.23.0.212,1222,172.23.0.10,445,Generic Protocol Command Decode,3,[1:2103003:7] GPL NETBIOS SMB-DS Session Setu...,TCP TTL:128 TOS:0x0 ID:851 IpLen:20 DgmLen:150...,***A**** Seq: 0x6B121C9E Ack: 0x72FF0D6E Win...,[Xref => http://www.microsoft.com/technet/secu...,NaN,NaN,NaN,2012-04-05 17:56:00


In [4]:
def ip_type(ip_addr):
    ip_addr_split = ip_addr.split('.')

    DNS_Root_Servers = ['198.41.0.4', '128.9.0.107', '192.33.4.12', '128.8.10.90', '192.203.230.10', '192.5.5.241', '192.112.36.4', '128.63.2.53', '192.36.148.17', '192.58.128.30', '193.0.14.129', '198.32.64.12', '202.12.27.33']
    if ip_addr in DNS_Root_Servers:
        return "DNS Root Server"
    
    website = False
    if ip_addr_split[0] == '10' and ip_addr_split[1] == '32':
        if ip_addr_split[2] == '0' and int(ip_addr_split[3]) >= 201 and int(ip_addr_split[3]) <= 210:
            website = True
        if ip_addr_split[2] == '1' and (ip_addr_split[3] == '100' or (int(ip_addr_split[3]) >= 201 and int(ip_addr_split[3]) <= 206)):
            website = True
        if ip_addr_split[2] == '5':
            website = True
    if website:
        return "website"
    
    if ip_addr == '10.32.0.1':
        return "Cisco ASA Firewall (external)"
    if ip_addr == '172.23.0.1':
        return "Cisco ASA Firewall (internal)"
    
    if ip_addr == '10.32.2.100' or ip_addr == '10.32.2.101':
        return 'SNAT'
    
    if ip_addr == '10.32.0.100':
        return 'Corporate Firewall (external)'
    
    if ip_addr == '172.25.0.1':
        return 'Corporate Firewall (internal)'
    
    # Regional Bank Network
    if ip_addr_split[0] == '172' and ip_addr_split[1] == '23':
        if int(ip_addr_split[2]) >= 214 and int(ip_addr_split[2]) <= 229:
            return 'Financial Server'
        if ip_addr == '172.23.0.2':
            return 'Log Server'
        if ip_addr == '172.23.0.10':
            return 'Domain Controller / DNS'
        else:
            return 'Workstation'
    
    return 'Other'

In [23]:
def label_source(row):
    return f"{ip_type(row[1])}"

def label_destination(row):
    return f"{ip_type(row[3])}"
def label_source_dest(row):
    return f"{ip_type(row[1])}---{ip_type(row[3])}"
def classification(row):
    return f"{row[5]}"

IDS_df['Source IP Type'] = IDS_df.apply(label_source, axis=1)
IDS_df['Dest IP Type'] = IDS_df.apply(label_destination, axis=1)
IDS_df['IP types'] = IDS_df.apply(label_source_dest, axis=1)
# IDS_df['classification'] = IDS_df.apply(classification, axis=1)
IDS_df.head()

,time,sourceIP,sourcePort,destIP,destPort,classification,priority,label,packet info,packet info cont'd,xref,label,packet info,packet info cont'd,Date/time,Source IP Type,Dest IP Type,IP types
0,4/5/2012 17:55,172.23.1.101,1101,172.23.0.10,139,Generic Protocol Command Decode,3,[1:2100538:17] GPL NETBIOS SMB IPC$ unicode s...,TCP TTL:128 TOS:0x0 ID:1643 IpLen:20 DgmLen:12...,***AP*** Seq: 0xCEF93F32 Ack: 0xC40C0BB Win:...,NaN,NaN,NaN,NaN,2012-04-05 17:55:00,Workstation,Domain Controller / DNS,Workstation---Domain Controller / DNS
1,4/5/2012 17:55,172.23.1.101,1101,172.23.0.10,139,Generic Protocol Command Decode,3,[1:2100538:17] GPL NETBIOS SMB IPC$ unicode s...,TCP TTL:128 TOS:0x0 ID:1649 IpLen:20 DgmLen:12...,***AP*** Seq: 0xCEF942A6 Ack: 0xC40C427 Win:...,NaN,NaN,NaN,NaN,2012-04-05 17:55:00,Workstation,Domain Controller / DNS,Workstation---Domain Controller / DNS
2,4/5/2012 17:55,172.23.1.101,1104,172.23.0.10,139,Generic Protocol Command Decode,3,[1:2103000:7] GPL NETBIOS SMB Session Setup N...,TCP TTL:128 TOS:0x0 ID:1663 IpLen:20 DgmLen:15...,***A**** Seq: 0x54FEF4C7 Ack: 0x9BEB6342 Win...,[Xref => http://www.microsoft.com/technet/secu...,NaN,NaN,NaN,2012-04-05 17:55:00,Workstation,Domain Controller / DNS,Workstation---Domain Controller / DNS
3,4/5/2012 17:55,172.23.1.101,1104,172.23.0.10,139,Generic Protocol Command Decode,3,[1:2100538:17] GPL NETBIOS SMB IPC$ unicode s...,TCP TTL:128 TOS:0x0 ID:1665 IpLen:20 DgmLen:12...,***AP*** Seq: 0x54FEFF59 Ack: 0x9BEB64C7 Win...,NaN,NaN,NaN,NaN,2012-04-05 17:55:00,Workstation,Domain Controller / DNS,Workstation---Domain Controller / DNS
4,4/5/2012 17:56,172.23.0.212,1222,172.23.0.10,445,Generic Protocol Command Decode,3,[1:2103003:7] GPL NETBIOS SMB-DS Session Setu...,TCP TTL:128 TOS:0x0 ID:851 IpLen:20 DgmLen:150...,***A**** Seq: 0x6B121C9E Ack: 0x72FF0D6E Win...,[Xref => http://www.microsoft.com/technet/secu...,NaN,NaN,NaN,2012-04-05 17:56:00,Workstation,Domain Controller / DNS,Workstation---Domain Controller / DNS


In [25]:
# frequency_IDS_df = IDS_df.groupby('time').size().reset_index(name='Num')
IDS_df_filtered = IDS_df.query("` classification`==' Potential Corporate Privacy Violation'")
IDS_df_filtered.loc[~((IDS_df_filtered['time']=='4/6/2012 17:28') | (IDS_df_filtered['time']=='4/6/2012 17:27') | (IDS_df_filtered['time']=='4/5/2012 18:07')),'time'] = "other"
fig = px.pie(IDS_df_filtered, names='time', title='Top times of Potential Corporate Privacy Violations')
fig.show()
# frequency_IDS_df.head()